# Monitoring variant: b210/oai-ran/<YOUR_PASSWORD>.ipynb

Based on the upstream example. The original remains under `examples/b210/oai-ran/<YOUR_PASSWORD>.ipynb`.
Core: **open5gs**; RAN: **oai**. Open5GS slice KPIs use the custom SMF/UPF image. The OAI gNB statistics parser is enabled.

# *5G deployment on already provisioned infrastructure*

### **Description**:
- #### Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR
- #### OAI gNB with USRP B210
- #### Quectel RM520N-GL (`qdon01`) as UE

Reference configuration: https://gitlab.eurecom.fr/oai/openairinterface5g/-/blob/develop/targets/PROJECTS/GENERIC-NR-5GC/CONF/gnb.sa.band78.fr1.106PRB.usrpb210.conf

## **0.**   Pre-requisites:

### Install Post-5G orchestrator:

In [ ]:
%pip install -e ../../../.. --extra-index-url=https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple

## Load input models:

- #### Meta Model:

In [ ]:
from metamodel.models.metamodel import MetaModel

metamodel = MetaModel.from_yaml(
"""
rus:
  b210:
    type: usrp
    model: b210
    txAttenuationDB: 10
    rxAttenuationDB: 10
    rxGainDB: 75

ran:
  components:
    gnb:
      type: gnb
      config:
        radioConfiguration:
          nrBand: 78
          pointAFrequencyArfcn: 640008
          channelBandwidthMhz: 40
          ssbArfcn: 641280
          tddConfiguration:
            periodicityMilliseconds: 5
            nrofDownlinkSlots: 7
            nrofDownlinkSymbols: 6
            nrofUplinkSlots: 2
            nrofUplinkSymbols: 4
        ru: b210

ues:
  qdong01:
    type: cots
    config:
      supi: "001010000000017"
      authenticationData:
        k: "fec86ba6eb707ed08905757b1bb44b8f"
        opc: "C42449363BBAD02B66D16BC975D77CC1"
      subscriptions:
        - snssai: internet
          dnn: internet
      gnbs:
        - gnb
      nrBands:
        - 78
"""
)

- #### Deployment Model:

In [ ]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
nodes:
  ptb-04015558.inria.fr: # node's actual hostname
    is_provisioned: true
    provisioner: null

  pacify.inria.fr: # node's actual hostname
    is_provisioned: true
    provisioner: null
    ssh_info:
      user: ubuntu
      sudo_password: ubuntu

rus:
  b210:
    is_provisioned: true
    provisioner: null

core:
  implementation: open5gs
  database_params:
    image:
      repository: "docker.io/mongo"
      image_tag: "7.0" # this is the mongodb version compatible with the machine's kernel version

ran:
  implementation: oai
  components:
    gnb:
      type: gnb
      advanced_config:
        initialDLBWPlocationAndBandwidth: 28875
        initialDLBWPcontrolResourceSetZero: 12
        initialULBWPlocationAndBandwidth: 28875
        ssb_perRACH_OccasionAndCB_PreamblesPerSSB: 14
        pusch_TargetSNRx10: 250
        pucch_TargetSNRx10: 200
        prach_dtx_threshold: 120
        pucch0_dtx_threshold: 100
        useAdditionalOptions: "-E --continuous-tx"
"""
)

## **1.**   Kubernetes Deployement:

In [ ]:
from orchestrator.deployment import K8SManagement

k8s = K8SManagement(metamodel=metamodel, deployment_model=deployment_model)

#### Deploy Kubernetes only on `ptb-04015558.inria.fr` machine:

In [ ]:
await k8s.deploy(nodes=["ptb-04015558.inria.fr"])

## **2.**   5G Core Deployement:

In [ ]:
from orchestrator.deployment import CoreManagement

core = CoreManagement(k8s)

In [ ]:
# Use the metrics image for SMF slice mappings and UPF traffic counters.
from metamodel.models.core import CoreComponentType
from orchestrator.models.k8s import ContainerImage
from orchestrator.models.core import ADVANCED_CORE_CONFIG_MAPPING, CoreImplementation, DeployedCoreComponent
from orchestrator.utils.defaults import DEFAULT_CORE_IMAGES

for component in core.deployment_model.core.components.values():
    if component.implementation != CoreImplementation.OPEN5GS:
        continue
    advanced_type = ADVANCED_CORE_CONFIG_MAPPING[component.implementation][component.type]
    advanced = component.advanced_config.model_dump() if component.advanced_config is not None else {}
    if component.type in (CoreComponentType.SMF, CoreComponentType.UPF):
        component.image = ContainerImage(
            repository="ghcr.io/niloysh/open5gs",
            image_tag="v2.7.0-upf-metrics-v2",
        )
        component.open5gs_binary_dir = "/open5gs/install/bin"
        advanced["logger"] = {"file": f"/tmp/open5gs-{component.type}.log", "level": "info"}
        if component.type == CoreComponentType.UPF:
            advanced.setdefault("teid_range_indication", 0)
    else:
        component.image = DEFAULT_CORE_IMAGES[component.implementation][component.type].model_copy(deep=True)
        component.open5gs_binary_dir = DeployedCoreComponent.model_fields["open5gs_binary_dir"].default
        if advanced.get("logger") == {"file": f"/tmp/open5gs-{component.type}.log", "level": "info"}:
            advanced.pop("logger")
    component.advanced_config = advanced_type.model_validate(advanced)

for name in ("smf", "upf"):
    image = core.deployment_model.core.components[name].image
    print(f"{name.upper()} image selected: {image.repository}:{image.image_tag}")

In [ ]:
await core.deploy()

### Check deployed pods with `kubectl`:

In [ ]:
from orchestrator.deployment import ProvisioningManagement

prov = ProvisioningManagement(core)

print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "open5gs-core"],
    nodes=["ptb-04015558.inria.fr"],
    become=True, quiet=True))["ptb-04015558.inria.fr"])

## Monitoring before UE attachment

The profile uses the control host selected by Kubernetes, including physical
Sopnodes. Capture interfaces refer to pod interfaces, not the host or radio NIC.
Keep the existing provisioning, RF and Multus settings. Grafana uses `admin` / `admin`.

In [ ]:
from pathlib import Path
import asyncio
import json

from orchestrator.deployment import MonitoringManagement
from orchestrator.models.monitoring import MonitoringDeployment

QUERIES_FILE = Path("queries.json")  # Your Prometheus export query profile.

monitoring = MonitoringManagement(preceding=ran)
monitoring.deployment_model.monitoring = MonitoringDeployment.from_yaml_file("monitoring.yaml")

In [ ]:
try:
    await monitoring.deploy()
finally:
    monitoring.output(
        metamodel_file="monitoring_metamodel.yaml",
        deployment_model_file="monitoring_deployment_model.yaml",
    )
prov = ProvisioningManagement(preceding=monitoring)

### Open Grafana

The SSH tunnel uses the current model address and jump host.
Open the link after deployment; close the tunnel with `grafana_tunnel.terminate()`.

In [ ]:
import atexit
import shlex
import subprocess
from IPython.display import Markdown, display

# Close the previous tunnel before opening one with the current host address.
if "grafana_tunnel" in globals() and grafana_tunnel.poll() is None:
    grafana_tunnel.terminate()
    await asyncio.to_thread(grafana_tunnel.wait, 5)

# get SSH info and fallback
cfg = monitoring.deployment_model.monitoring
control = cfg.installed_control_node or cfg.control_node or monitoring.deployment_model.params.k8s.master_node
ssh = monitoring.deployment_model.nodes[control].ssh_info

command = ["ssh", "-N", "-T", "-S", "none", "-o", "ControlMaster=no", "-o", "ControlPersist=no",
           "-o", "BatchMode=yes", "-o", "ExitOnForwardFailure=yes", "-o", "ConnectTimeout=10"]

command += shlex.split(ssh.common_args or "")

if ssh.private_key_file:
    command += ["-i", str(Path(ssh.private_key_file).expanduser())]
command += ["-p", str(ssh.port), "-L", f"127.0.0.1:30096:127.0.0.1:{cfg.grafana_node_port}",
            f"{ssh.user}@{ssh.hostname or control}"]

grafana_tunnel = subprocess.Popen(command, stdout=subprocess.DEVNULL)

# Close the tunnel when the kernel exits normally.
atexit.register(grafana_tunnel.terminate)

GRAFANA_URL = "http://127.0.0.1:30096"
display(Markdown(f"[Open Grafana]({GRAFANA_URL}) — username: **admin**, password: **admin**"))

## **3.**   RAN deployement:

### Check if USRP B210 is plugged in on `ptb-04015558.inria.fr`:

In [ ]:
print((await prov.run_node_command(
    command=["lsusb"],
    nodes=["ptb-04015558.inria.fr"],
    quiet=True))["ptb-04015558.inria.fr"])

#### Tune `ptb-04015558` (reference: https://kb.ettus.com/USRP_Host_Performance_Tuning_Tips_and_Tricks):

In [ ]:
print((await prov.run_node_command(
    command=["apt", "install", "-y", "cpufrequtils"],
    nodes=["ptb-04015558.inria.fr"],
    become=True, quiet=True))["ptb-04015558.inria.fr"])

#### Set the CPU governor to performance for all cores:

In [ ]:
print((await prov.run_node_command(
    command=["bash", "-c", "for ((i=0;i<$(nproc --all);i++)); do cpufreq-set -c $i -r -g performance; done"],
    nodes=["ptb-04015558.inria.fr"],
    become=True, quiet=True))["ptb-04015558.inria.fr"])

#### Verify that the CPU governor has been set:

In [ ]:
print((await prov.run_node_command(
    command=["cpufreq-info"],
    nodes=["ptb-04015558.inria.fr"],
    become=True, quiet=True))["ptb-04015558.inria.fr"])

#### Deploy OAI gNB:

In [ ]:
from orchestrator.deployment import RANManagement

ran = RANManagement(core)

In [ ]:
# Enable the statistics-file exporter before the gNB pod is created.
from orchestrator.models.ran import OAIAdvancedGNBConfig

gnb = ran.deployment_model.ran.components["gnb"]
gnb.advanced_config = OAIAdvancedGNBConfig.model_validate(
    gnb.advanced_config.model_dump() if gnb.advanced_config is not None else {}
)
gnb.advanced_config.metrics_parser.enabled = True

In [ ]:
await ran.deploy()

### Check gNB logs:

In [ ]:
print(await ran.get_logs(quiet=True))

## **3.**   Connect UE:

### Check if `qdong01` is plugged in on `pacify.inria.fr`:

In [ ]:
print((await prov.run_node_command(
    command=["lsusb"],
    nodes=["pacify.inria.fr"],
    quiet=True))["pacify.inria.fr"])

### Initialize UE:

In [ ]:
print((await prov.run_node_command(
    command=["qhat-init", "--dnn", "internet"],
    nodes=["pacify.inria.fr"],
    become=True, quiet=True))["pacify.inria.fr"])

> *Note*: If the command above fails with an error like this:
> ```bash
> [2026-10-05 04:21:40] 🔧 qhat-init — multi-modem init
> [2026-10-05 04:21:42] ❌ No modem AT port found.
> ```
> Then uncomment and run the `stop.sh` command and the `qhat-init` command again::

In [ ]:
# print((await prov.run_node_command(
#     command=["stop.sh"],
#     nodes=["pacify.inria.fr"],
#     become=True, quiet=True))["pacify.inria.fr"])

In [ ]:
# print((await prov.run_node_command(
#     command=["qhat-init", "--dnn", "internet"],
#     nodes=["pacify.inria.fr"],
#     become=True, quiet=True))["pacify.inria.fr"])

### Connect UE:

In [ ]:
print((await prov.run_node_command(
    command=["start.sh", "-F", "internet"],
    nodes=["pacify.inria.fr"],
    become=True, quiet=True))["pacify.inria.fr"])

### Check if UE got a PDU session:

In [ ]:
output = (await prov.run_node_command(
    command=["ip", "-4", "-o", "addr", "show", "wwan0"],
    nodes=["pacify.inria.fr"],
    quiet=True))["pacify.inria.fr"]

output

#### Get UE IP:

In [ ]:
ip = output.split()[3].split("/")[0]
ip

#### Check gNB logs:

In [ ]:
print(await ran.get_logs(quiet=True))

### Test connectivity:

In [ ]:
print((await prov.run_node_command(
    command=["ping", "-c", "6", "11.1.1.1", "-I", "wwan0"],
    nodes=["pacify.inria.fr"],
    quiet=True))["pacify.inria.fr"])

In [ ]:
print((await prov.run_node_command(
    command=["ping", "-c", "6", "8.8.8.8", "-I", "wwan0"],
    nodes=["pacify.inria.fr"],
    quiet=True))["pacify.inria.fr"])

## **4.**   Test Scenario: Iperf

#### Install `iperf3` on both machines:

In [ ]:
await prov.run_node_command(command=["sh", "-c", "DEBIAN_FRONTEND=noninteractive apt install -y iperf3"], become=True, quiet=True)

#### Add necessary routing rules for the UE on `pacify.inria.fr` machine:

In [ ]:
await prov.run_node_command(
    command=["ip", "route", "add", "11.1.1.0/29", "dev", "wwan0", "src", ip, "table", "100"],
    nodes=["pacify.inria.fr"],
    become=True, quiet=True,
)

In [ ]:
await prov.run_node_command(
    command=["ip", "route", "add", "default", "via", "11.1.1.1", "dev", "wwan0", "src", ip, "table", "100"],
    nodes=["pacify.inria.fr"],
    become=True, quiet=True,
)

In [ ]:
await prov.run_node_command(
    command=["ip", "rule", "add", "from", f"{ip}/32", "table", "100"],
    nodes=["pacify.inria.fr"],
    become=True, quiet=True,
)

## Iperf with experiment artifacts

The server runs on the Kubernetes control host from the deployment model.
Each client binds to its own 5G interface. Review the explicit scenario before running it.
The collector exports Prometheus data after traffic and saves each report with the timeline.

In [ ]:
from datetime import UTC, datetime
from orchestrator.artifacts import ExperimentRun
from orchestrator.deployment import ProvisioningManagement
from orchestrator.utils.ansible import get_ansible_inventory_line, run_ansible_command, run_ansible_playbook

SERVER_NODE = monitoring.deployment_model.params.k8s.master_node
CLIENTS = ['pacify.inria.fr']
UE_INTERFACE = 'wwan0'
DURATION, BASE_PORT = 20, 5202
BASELINE_SECONDS, COLLECTION_SECONDS = 5, 5

server_node = monitoring.deployment_model.nodes[SERVER_NODE]
server_ip = server_node.ssh_info.hostname or SERVER_NODE
server_inventory = get_ansible_inventory_line(SERVER_NODE, server_node)
server_runner = str(Path(monitoring.deployment_model.params.ansible.runner_path) / "iperf-server")
clients = ProvisioningManagement(preceding=monitoring)

await run_ansible_playbook(
    playbook=[{
        "name": "Install iperf3 on the server host", "hosts": SERVER_NODE,
        "gather_facts": False, "become": True,
        "tasks": [{"ansible.builtin.apt": {"name": "iperf3", "state": "present",
                  "update_cache": True, "cache_valid_time": 3600}}],
    }],
    inventory=server_inventory, runner_path=server_runner, quiet=True,
)
await clients.run_node_command(
    command=["sh", "-c", "DEBIAN_FRONTEND=noninteractive apt-get update && DEBIAN_FRONTEND=noninteractive apt-get install -y iperf3"],
    nodes=CLIENTS, become=True, quiet=True,
)
addresses = await clients.run_node_command(
    command=["ip", "-j", "-4", "addr", "show", "dev", UE_INTERFACE],
    nodes=CLIENTS, quiet=True,
)
ue_ips = {}
for client in CLIENTS:
    ipv4 = [a["local"] for device in json.loads(addresses[client]) for a in device["addr_info"]
            if a["family"] == "inet"]
    if len(ipv4) != 1:
        raise RuntimeError(f"{client}: expected one IPv4 address on {UE_INTERFACE}")
    ue_ips[client] = ipv4[0]
print(f"Server: {SERVER_NODE} ({server_ip}); UE clients: {ue_ips}")

In [ ]:
async def start_iperf_server(port):
    await run_ansible_command(
        command=["iperf3", "-s", "-D", "-1", "--idle-timeout", "60",
                 "-B", server_ip, "-p", str(port), "-J"],
        hosts=SERVER_NODE, inventory=server_inventory, runner_path=server_runner, quiet=True,
    )


async def iperf_client(client, direction, duration, port):
    reverse = {"uplink": [], "downlink": ["-R"]}[direction]
    output = await clients.run_node_command(
        command=["iperf3", "-c", server_ip, "-B", ue_ips[client], "--bind-dev", UE_INTERFACE,
                 "-p", str(port), "-t", str(duration), "--connect-timeout", "10000",
                 "-J", "--get-server-output", *reverse],
        nodes=[client], become=True, quiet=True,
    )
    report = json.loads(output[client])
    if report.get("error"):
        raise RuntimeError(report["error"])
    return report

In [ ]:
SCENARIO = [
    {"name": "ue1_uplink", "client": 'pacify.inria.fr', "direction": "uplink", "duration": DURATION},
    {"name": "ue1_downlink", "client": 'pacify.inria.fr', "direction": "downlink", "duration": DURATION},
]

In [ ]:
timestamp = datetime.now(UTC).strftime("%Y-%m-%d_%H-%M-%S.%fZ")
output_directory = Path("results") / f"iperf-{timestamp}"

async with ExperimentRun(
    output_directory,
    name="ue-to-host-iperf",
    monitoring=monitoring,
    queries_file=QUERIES_FILE,
) as experiment:
    experiment.save_json("scenario.json", SCENARIO)

    with experiment.section("baseline"):
        await asyncio.sleep(BASELINE_SECONDS)

    for port, <YOUR_PASSWORD> in enumerate(SCENARIO, start=BASE_PORT):
        print(f"Running {<YOUR_PASSWORD>['name']}: {<YOUR_PASSWORD>['client']}, {<YOUR_PASSWORD>['direction']}, {<YOUR_PASSWORD>['duration']} seconds")
        await start_iperf_server(port)

        with experiment.section(<YOUR_PASSWORD>["name"]):
            report = await iperf_client(
                client=<YOUR_PASSWORD>["client"],
                direction=<YOUR_PASSWORD>["direction"],
                duration=<YOUR_PASSWORD>["duration"],
                port=port,
            )

        experiment.save_json(f"{<YOUR_PASSWORD>['name']}.json", report)

    with experiment.section("collection"):
        await asyncio.sleep(COLLECTION_SECONDS)

print(f"Artifacts saved to: {output_directory.resolve()}")

### Save meta model and deployment model:

In [ ]:
ran.output(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

## Cleanup:

- #### Disconnect UE:

In [ ]:
# from orchestrator.deployment import ProvisioningManagement

# prov = ProvisioningManagement(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

# print((await prov.run_node_command(
#     command=["stop.sh"],
#     nodes=["pacify.inria.fr"],
#     become=True, quiet=True))["pacify.inria.fr"])

- #### Remove RAN:

In [ ]:
# from orchestrator.deployment import RANManagement

# ran = RANManagement(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

# await ran.remove()

- #### Remove 5G core:

In [ ]:
# from orchestrator.deployment import CoreManagement

# core = CoreManagement(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

# await core.remove()

- #### Remove Kubernetes:

In [ ]:
# from orchestrator.deployment import K8SManagement

# k8s = K8SManagement(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

# await k8s.remove()